# Step 5: Model Deployment

Deploy the trained model for inference. Snowflake supports **two deployment approaches**:

| Aspect | SQL Inference | REST Inference |
|--------|--------------|----------------|
| **How to Access** | SQL queries inside Snowflake | HTTP requests from external apps |
| **Best For** | Batch processing on tables | Real-time single predictions |
| **Latency** | Higher (warehouse startup) | Lower (always-on service) |
| **Use Case** | ETL pipelines, scheduled jobs | Web apps, mobile apps, APIs |
| **Scaling** | Warehouse size | SPCS auto-scaling |
| **Cost Model** | Warehouse credits | SPCS compute pool |

## Prerequisites

- Run notebooks 01-05 first

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config
from source.utils import get_session, get_feature_config
from source.framework.deploy import ModelDeployer

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Get Model Version from Registry

In [ ]:
MODEL_NAME = config.model.model_name
COMPUTE_POOL = config.compute.compute_pool
SERVICE_NAME = config.deploy.service_name
test_table = f"{DB}.{SCHEMA}.{config.tables.test_features}"

deployer = ModelDeployer(
    session=session,
    registry_database=DB,
    registry_schema=SCHEMA,
)

if not MODEL_VERSION:
    MODEL_VERSION = deployer.get_latest_version_name(MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print(f"Version to deploy: {MODEL_VERSION}")
print(f"Service: {SERVICE_NAME}")
print(f"Compute pool: {COMPUTE_POOL}")

In [ ]:
if deployer.service_exists(SERVICE_NAME):
    print(f"Existing service found: {SERVICE_NAME}")
    print(f"  Status: {deployer.get_service_status(SERVICE_NAME)}")
else:
    print(f"No existing service '{SERVICE_NAME}'")

## Option 1: SQL Inference Deployment

Deploy the model for SQL inference by setting the default version. This allows calling `MODEL!PREDICT()` directly in SQL queries.

**Use when**: Batch processing on Snowflake tables, scheduled ETL jobs, data pipelines

### Set Default Model Version

In [ ]:
deployer.set_default_version(MODEL_NAME, MODEL_VERSION)
print(f"Default version set: {MODEL_VERSION}")
print(f"SQL endpoint: {DB}.{SCHEMA}.{MODEL_NAME}!PREDICT()")

### Test SQL Inference

Run inference using SQL on sample data from the test table.

In [ ]:
feature_config = get_feature_config(config)
feature_columns = [c.upper() for c in feature_config["all_numeric_features"] + feature_config["all_categorical_features"]]

named_args = ", ".join(f"{c} => {c}" for c in feature_columns)
sql_query = f"""
SELECT
    AGE, GENDER, ADMISSION_TYPE,
    RISK_LEVEL AS ACTUAL_RISK,
    {DB}.{SCHEMA}.{MODEL_NAME}!PREDICT(
        {named_args}
    ):output_feature_0::STRING AS PREDICTED_RISK
FROM {test_table}
LIMIT 5
"""

print("SQL Inference Results:")
session.sql(sql_query).show()

## Option 2: REST Inference Deployment (SPCS Service)

Deploy the model as a containerized REST endpoint on Snowpark Container Services for real-time inference from external applications.

**Use when**: Web/mobile apps, external API integrations, low-latency real-time predictions

**Note**: Service creation takes 5-10 minutes on first deployment.

### Create Inference Service

In [ ]:
if deployer.service_exists(SERVICE_NAME):
    print(f"Dropping existing service '{SERVICE_NAME}'...")
    deployer.drop_service(SERVICE_NAME)

print(f"Deploying {MODEL_NAME}/{MODEL_VERSION} as service '{SERVICE_NAME}'...")
deployer.deploy(
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    service_name=SERVICE_NAME,
    compute_pool=COMPUTE_POOL,
    min_instances=config.deploy.min_instances,
    max_instances=config.deploy.max_instances,
)
print(f"Service '{SERVICE_NAME}' is RUNNING")

### Validate REST Inference

Run sample predictions through the deployed service via the Model Registry.

In [ ]:
import pandas as pd

sample_df = session.table(test_table).limit(3).to_pandas()
sample_df.columns = [c.upper() for c in sample_df.columns]

predictions = deployer.predict(
    model_name=MODEL_NAME,
    service_name=SERVICE_NAME,
    version_name=MODEL_VERSION,
    features_df=sample_df[feature_columns],
    function_name="predict",
)

result_df = sample_df[["PATIENT_ID", "RISK_LEVEL"]].copy()
result_df["PREDICTED_RISK"] = predictions["output_feature_0"].values

print("Sample predictions via deployed service:")
display(result_df)

## Check Service Status

In [ ]:
status = deployer.get_service_status(SERVICE_NAME)
if status:
    print(f"Service: {SERVICE_NAME}")
    print(f"  Status: {status}")
else:
    print("Service not found")

## Next Step

Continue to **06_model_monitoring.ipynb**